# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [66]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [67]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [68]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [69]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [ ]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful, knowledgeable, and professional health and wellness expert. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [ ]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [ ]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on your hands and knees. Alternate between arching your back upward (like a cat) and letting it sag downward (like a cow). Aim for 10-15 repetitions to improve flexibility and relieve tension.\n\n2. **Bird Dog**: From your hands and knees, extend opposite arm and leg simultaneously while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Perform 10 repetitions per side to strengthen lower back and core muscles.\n\n3. **Pelvic Tilts**: Lie on your back with knees bent and feet flat on the floor. Tighten your abdominal muscles and tilt your pelvis slightly upward, pressing your lower back into the floor. Hold for 10 seconds and repeat 8-12 times to help stabilize your lower back.\n\n4. **Partial Crunches**: Lie on your back with knees bent, cross arms over your chest, and gently lift your shoulders off the floor using your abdominal muscles. Lower back down carefull

In [ ]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in maintaining overall health by supporting various bodily and mental functions. Adequate quality sleep (typically 7-9 hours for adults) allows the body to repair tissues, regenerate cells, and regulate hormones essential for growth and appetite. During sleep, particularly in the deep and REM stages, the brain consolidates memories and processes learning, which is crucial for cognitive health.\n\nFurthermore, sleep is instrumental in strengthening the immune system, making the body more resilient against illnesses. Poor sleep or sleep disturbances like insomnia can negatively impact mental well-being, increase the risk of chronic conditions such as heart disease, diabetes, and obesity, and contribute to issues like stress and mood disorders.\n\nPracticing good sleep hygiene—such as maintaining a consistent sleep schedule, creating a conducive sleep environment, and establishing relaxing routines before bed—can significantly enhance sleep quality and, consequen

In [ ]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- **For Headaches:**\n  - Drink plenty of water to stay hydrated.\n  - Apply cold or warm compresses to the head or neck.\n  - Rest in a dark, quiet room.\n  - Gently massage the temples and neck.\n  - Use essential oils such as peppermint or lavender.\n  - Maintain a regular sleep schedule.\n  - Be mindful of common triggers like dehydration, stress, poor sleep, skipped meals, eye strain, weather changes, and certain foods.\n\n- **For Stress Relief:**\n  - Practice deep breathing exercises, such as inhaling for 4 counts, holding for 4, and exhaling for 4.\n  - Engage in progressive muscle relaxation, tensing and releasing muscle groups from toes to head.\n  - Use grounding techniques by identifying things you see, hear, feel, smell, and taste.\n  - Take short walks, preferably in nature.\n  - Listen to calming music.\n  - Briefly splash cold water on your face or wrists.\n  - Practice mindfulness and meditation regularly.\n  

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [ ]:
# Adding install
!pip install rank-bm25

In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [ ]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [ ]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on your hands and knees. Arch your back upwards (cat) and then let it sag downwards (cow). Perform 10-15 repetitions to promote flexibility and relieve tension.\n\n2. **Bird Dog**: From your hands and knees, extend opposite arm and leg simultaneously while engaging your core. Hold each stretch for about 5 seconds, and repeat 10 times on each side to strengthen your back and improve stability.\n\n3. **Pelvic Tilts**: Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting the pelvis slightly upward. Hold for 10 seconds, and repeat 8-12 times to help improve lower back flexibility and strength.\n\nThese gentle stretching and strengthening exercises can help alleviate discomfort and potentially prevent future episodes of lower back pain. However, if you have persistent or severe pain, it is important to consult a healthcare profession

In [ ]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Sleep plays a vital role in overall health and wellness. Maintaining a consistent sleep schedule and creating an optimal sleep environment—such as keeping the room cool, dark, and quiet—are key practices that promote quality sleep. Good sleep hygiene, including establishing relaxing bedtime routines, limiting screen time before bed, and avoiding caffeine in the late afternoon or evening, can significantly improve sleep quality. Adequate sleep supports immune function, mental health, nutrient absorption, and overall energy levels. Conversely, poor or insufficient sleep can negatively impact physical health, mental clarity, mood, and the body's ability to recover and function effectively."

In [ ]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Some natural remedies for stress and headaches include:\n\n- **Relaxation Techniques**: Practices such as deep breathing exercises, progressive muscle relaxation, and meditation can help reduce stress and muscle tension that may lead to headaches.\n- **Herbal Teas**: Chamomile and valerian root teas are known for their calming effects and can promote relaxation.\n- **Adequate Hydration**: Drinking plenty of water helps prevent dehydration, a common trigger for headaches.\n- **Proper Sleep Habits**: Establishing a regular sleep schedule and practicing calming bedtime routines can improve sleep quality and reduce headaches related to sleep problems.\n- **Balanced Diet**: Eating regular, nutritious meals and avoiding known trigger foods like alcohol, processed meats, and aged cheese can help prevent headaches.\n- **Stress Management**: Engaging in activities that promote mental well-being, such as mindfulness or gentle exercises, can help manage chronic stress which contributes to headac

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

Example query: “What does section 5.2.1 of POLICY-SEC-042 say about rotating AWS access keys?”

BM25 is the perfect choice for the query because it's far better with exact wording, IDs, numbers, or exact phrases than embeddings are. Embeddings are much better at semantic similarity, synonyms, paraphrases, and 'fuzzy'/unclear/inaccurate queries.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [ ]:
!pip install -U langchain-classic

In [ ]:
# Updating these import packages
from langchain_cohere import CohereRerank
from langchain_core.documents import Document
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [ ]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"To help alleviate lower back pain, gentle stretching and strengthening exercises are recommended. Some effective exercises include:\n\n1. **Cat-Cow Stretch:**  \n- Start on your hands and knees.  \n- Arch your back upward (cat pose), hold for a few seconds, then let it sag down (cow pose).  \n- Repeat this movement 10-15 times.\n\n2. **Bird Dog:**  \n- Begin on hands and knees.  \n- Extend your opposite arm and leg simultaneously while keeping your core engaged.  \n- Hold the position for about 5 seconds, then switch sides.  \n- Do 10 repetitions per side.\n\n3. **Pelvic Tilts:**  \n- Lie on your back with knees bent and feet flat on the floor.  \n- Tighten your abdominal muscles and tilt your pelvis upward, pressing your lower back into the floor.  \n- Hold for 10 seconds, then relax.  \n- Repeat 8-12 times.\n\nThese exercises can help strengthen the muscles supporting your lower back and improve flexibility, which can reduce discomfort and prevent future issues. However, if you expe

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health in multiple ways. It is essential for physical recovery, as during sleep, the body repairs tissues and regenerates cells. Sleep also supports mental well-being by consolidating memories and facilitating learning. Moreover, it helps regulate hormones that control growth and appetite, influencing weight management and metabolic health. Adequate sleep, typically 7-9 hours per night for adults, is crucial for maintaining good health, cognitive function, and emotional stability. Poor or insufficient sleep can lead to a range of health issues, including weakened immune function, increased risk of chronic diseases, cognitive impairments, and emotional disturbances.'

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying a cold or warm compress to your head or neck\n- Resting in a dark, quiet room\n- Performing gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing deep breathing exercises\n- Engaging in progressive muscle relaxation\n- Using grounding techniques (e.g., noticing things around you to calm your mind)\n- Taking short walks, preferably in nature\n- Listening to calming music\n\nThese methods can help alleviate stress and reduce headache symptoms naturally.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [ ]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [ ]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n1. **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n2. **Bird Dog:** From hands and knees, extend opposite arm and leg while engaging your core. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. **Partial Crunches:** Lie on your back with knees bent, arms crossed over your chest, tighten your stomach muscles, and lift your shoulders off the floor briefly before lowering back down. Do 8-12 repetitions.\n\n4. **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold the position for 15-30 seconds, then switch legs.\n\n5. **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by engaging core muscles and tilting your pelvis upward. Hold for about 10 seconds and repeat 8-12 times.\n\nRemember to perform the

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health, impacting physical, mental, and emotional well-being. According to the provided information, sleep is crucial for tissue repair, hormone regulation, memory consolidation, and immune function. Adequate sleep (7-9 hours per night) helps the body recover from daily stressors, supports brain health, and enhances immune defenses. Poor sleep or sleep disruptions can lead to negative health effects such as increased fatigue, stress, immune suppression, and higher risk for chronic conditions. Maintaining good sleep hygiene practices—like sticking to a consistent schedule, creating a restful sleep environment, and avoiding screens and stimulants before bed—can significantly improve sleep quality and thus promote overall health.'

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Natural remedies for stress and headaches include several effective strategies:\n\n1. **Hydration:** Drinking plenty of water helps prevent dehydration, which can trigger headaches and contribute to stress.\n\n2. **Relaxation Techniques:**\n   - Deep breathing exercises (e.g., inhaling for 4 counts, holding, exhaling for 4 counts)\n   - Progressive muscle relaxation, tensing and releasing muscle groups from toes to head\n   - Mindfulness and meditation to promote present-moment awareness and reduce stress\n\n3. **Use of Essential Oils:** Peppermint and lavender oils can be applied or inhaled to help alleviate headache pain and promote relaxation.\n\n4. **Headache-specific remedies:**\n   - Applying warm or cold compresses to the head or neck\n   - Resting in a dark, quiet room\n   - Gentle massage of the temples and neck\n\n5. **Lifestyle and Environment:**\n   - Maintaining a regular sleep schedule and quality sleep routine\n   - Managing stress through hobbies, social support, and s

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Generating multiple reformulations of a user query improves recall because each version uses different words, structures, and levels of specificity which results in the retrieval of more and different documents than a single phrase retreival ever could. Some formulations bring to surface synonym-heavy matches, others surface ID-like terms or narrow subquestions. When the top results are merged across all these queries, many more relevant documents in the corpus are coverd, so the RAG step has richer and more complete context to answer from.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [ ]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [ ]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [ ]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [ ]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Based on the provided information, exercises that can help alleviate lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Aim for 10-15 repetitions.\n\n2. **Bird Dog**: From hands and knees, extend opposite arm and leg while engaging your core. Hold each extension for 5 seconds and switch sides. Perform 10 repetitions per side.\n\n3. **Partial Crunches**: Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor briefly before lowering. Do 8-12 repetitions.\n\n4. **Knee-to-Chest Stretch**: Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds and switch legs.\n\n5. **Pelvic Tilts**: Lie on your back with knees bent, tighten your abdominal muscles, and tilt your pelvis upward to flatten your back against the floor. Hold for 10 seconds and repeat

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health in multiple ways. Adequate sleep of 7-9 hours per night allows the body to repair tissues, consolidate memories, and regulate hormones that influence growth and appetite. During different sleep stages—particularly deep sleep (Stage 3) and REM sleep—the body repairs tissues, strengthens the immune system, and supports cognitive functions like learning and memory.\n\nPoor or insufficient sleep can lead to fatigue, decreased concentration, weakened immune response, mood disturbances, and increased risk of chronic conditions such as heart disease, diabetes, and obesity. Additionally, disrupted sleep cycles or conditions like insomnia can impair mental well-being and overall quality of life.\n\nPracticing good sleep hygiene—such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and optimizing the sleep environment—can enhance sleep quality and, consequently, overall health.'

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Natural remedies for stress and headaches include several gentle and effective options:\n\n1. Hydration: Drinking plenty of water helps prevent dehydration, which can trigger headaches and increase stress levels.\n2. Relaxation techniques: Practices such as deep breathing exercises, progressive muscle relaxation, and mindfulness meditation can reduce stress and promote physical and mental calmness.\n3. Essential oils: Peppermint and lavender essential oils are known for their soothing properties. Applying diluted peppermint oil to the temples or using lavender oil in a diffuser can help alleviate headaches and promote relaxation.\n4. Gentle massage: Massaging the temples, neck, and shoulders can relieve muscle tension and reduce headache severity.\n5. Rest in a conducive environment: Resting in a dark, quiet room helps manage headaches and provides mental relief.\n6. Warm or cold compresses: Applying a warm compress can relax tense muscles, while a cold compress can numb pain and redu

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [ ]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n1. **Cat-Cow Stretch:** Start on your hands and knees. Arch your back upwards (cat) and then let it sag downwards (cow). Perform 10-15 repetitions to improve flexibility and reduce tension.\n\n2. **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Aim for 10 repetitions per side to strengthen the lower back and improve stability.\n\n3. **Pelvic Tilts:** Lie on your back with knees bent, flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times to relieve tension and improve pelvic mobility.\n\n4. **Partial Crunches:** Lie on your back with knees bent, cross your arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor briefly before lowering back down. Do 8-12 repetitions to strengthen 

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health in several ways. During sleep, the body repairs tissues, consolidates memories, and regulates hormones that influence growth and appetite. Adequate sleep—typically 7 to 9 hours per night—supports physical health, mental well-being, and cognitive function. It helps strengthen the immune system, reduce stress, improve mood, and enhance learning and memory. Conversely, poor sleep or sleep deprivation can lead to increased risk of chronic conditions such as heart disease, diabetes, and mental health issues like anxiety and depression. Maintaining good sleep hygiene, creating a restful environment, and managing sleep disorders like insomnia are crucial steps toward promoting overall wellness.'

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., inhale for 4 counts, hold for 4, exhale for 4)\n- Progressive muscle relaxation, tensing and releasing muscle groups from toes to head\n- Grounding techniques, such as identifying things you see, hear, feel, smell, and taste\n- Gentle massage of temples and neck\n- Using essential oils like peppermint or lavender\n- Applying warm or cold compresses to the head or neck\n- Maintaining a regular sleep schedule and creating a relaxing bedtime routine\n- Staying well-hydrated by drinking water\n- Engaging in calming activities such as listening to soothing music or taking short walks, especially in nature\n\nManaging stress and headaches holistically often involves combining these natural remedies with overall lifestyle practices like adequate sleep, balanced nutrition, stress management techniques, and regular physical activity.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [ ]:
!pip install langchain-experimental

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [ ]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [ ]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [ ]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [ ]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [ ]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Begin on hands and knees. Alternately arch your back upward (cat pose) and let it sag downward (cow pose). Perform 10-15 repetitions to increase flexibility and reduce tension.\n\n2. **Partial Crunches**: Lie on your back with knees bent and arms crossed over your chest. Tighten your abdominal muscles and lift your shoulders off the floor, then lower. Do 8-12 repetitions to strengthen core muscles.\n\n3. **Knee-to-Chest Stretch**: Lie on your back and pull one knee toward your chest while keeping the other foot flat on the ground. Hold for 15-30 seconds, then switch legs to stretch lower back muscles.\n\n4. **Pelvic Tilts**: Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times to improve pelvic stability and reduce back discomfort.\n\n5. **Bird Dog**: From hands and knees, 

In [ ]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health and wellness. According to the provided information, sleep affects physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults generally need 7-9 hours of sleep per night, with sleep occurring in cycles that include REM and non-REM stages, each contributing differently to health. Good sleep hygiene practices—such as maintaining a consistent schedule, creating a relaxing bedtime routine, and optimizing the sleep environment—are essential for quality sleep. Furthermore, issues like insomnia and poor sleep habits can negatively impact physical and mental health, increasing the risk of conditions such as fatigue, headaches, impaired immune function, and mental health problems like anxiety and depression. Therefore, adequate, high-quality sleep is crucial for maintaining overall health, supporting immune function, mental c

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

Percentile-based semantic chunking's algorithm measures embedding similarity between adjacent sentences and cuts where similarity drops past a selected percentile. In the case of FAQs with short, repetitive sentences, similarities span a tight band or narrow range, so the percentile threshold stops reflecting topic shifts resulting in almost no breakpoints and a huge grab-bag chunks of many FAQs instead, or splits at completely arbitrary and very much non-semantic points. Retrieval under these conditions will return big mixed chunks instead of the ideal one or two entries.

Improvements/adjustments

Add structural rules-
Treat each FAQ entry (question + answer) as a base unit and only merge neighboring entries when similarity is extremely high.

Tighten thresholds-
Set a lower percentile or an absolute similarity cutoff instead, tuned to this data. Also employ a more strict max chunk size, so chunk size can be kept near one or two or a few FAQs, instead of entire pages.



In [ ]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, grounding techniques (such as identifying objects around you through your senses), listening to calming music, and engaging in short walks, especially in nature. Practicing mindfulness and meditation regularly—such as focusing on your breath or doing body scans—can also significantly reduce stress levels.\n\nFor headaches, natural remedies include staying well-hydrated by drinking plenty of water, applying cold or warm compresses to the head or neck, resting in a dark and quiet environment, massaging temples and neck muscles gently, and using essential oils like peppermint or lavender. Additionally, maintaining a regular sleep schedule, managing stress effectively, and avoiding common triggers such as certain foods or weather changes can help prevent and alleviate headaches.\n\nIf you're considering herbal teas, chamomile and valerian root are known for their calming effects and may assis

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### MY CODE HERE

# Golden dataset was created using Ragas TestsetGenerator in the 
# generate_golden_dataset.py script in the project folder. 
# Generate_golden_dataset.py writes to the golden_dataset.json file, 
# also found in the project folder. This was done to prevent the many ragas 
# failures from infecting and catestrophically hanging the notebook. 

!pip install -q ragas

In [ ]:
# Loading the golden dataset! 

import json
import pandas as pd

with open("golden_dataset.json", "r", encoding="utf-8") as f:
    golden_records = json.load(f)

questions  = [r["user_input"] for r in golden_records]
references = [r["reference"] for r in golden_records]

print(f"Loaded {len(questions)} questions from golden_dataset.json")
pd.DataFrame(golden_records).head(5)

Loaded 18 questions from golden_dataset.json


,user_input,reference
0,What is Personal Wellness Guide and how it hel...,The Personal Wellness Guide is a comprehensive...
1,Wht is the benfit of excersize?,Exercise is one of the most important things y...
2,What are macronutriants and why are they impor...,Macronutrients are key components of a balance...
3,What are vitimins and why are they important f...,Vitamins are organic compounds needed in small...
4,What are some good things to do for Evening Wi...,Effective evening routine elements include set...


In [ ]:
# Running each of the five retrievers over the golden dataset - boom!
import time
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

def make_chain(retriever):
    return (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=itemgetter("context"))
        | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
    )

retriever_map = {
    "Naive":               naive_retriever,
    "BM25":                bm25_retriever,
    "Multi-Query":         multi_query_retriever,
    "Parent-Document":     parent_document_retriever,
    "Contextual-Compress": compression_retriever,
}

PRICE_INPUT_PER_TOK  = 0.15 / 1_000_000
PRICE_OUTPUT_PER_TOK = 0.60 / 1_000_000
# gpt-4o-mini pricing: $0.15 per 1M input tokens, $0.60 per 1M output tokens

run_results = {}

for name, retriever in retriever_map.items():
    chain = make_chain(retriever)
    answers, contexts, latencies, costs = [], [], [], []

    for q in questions:
        t0  = time.perf_counter()
        out = chain.invoke({"question": q})
        latencies.append(time.perf_counter() - t0)

        answers.append(out["response"].content)
        contexts.append([doc.page_content for doc in out["context"]])

        usage = out["response"].response_metadata.get("token_usage", {})
        costs.append(
            usage.get("prompt_tokens", 0)     * PRICE_INPUT_PER_TOK +
            usage.get("completion_tokens", 0) * PRICE_OUTPUT_PER_TOK
        )
        if name == "Contextual-Compress":
            time.sleep(7)
        # ─────────────────────────────────────────────────────────────────────

    run_results[name] = {
        "answers":  answers,
        "contexts": contexts,
        "avg_lat":  round(sum(latencies) / len(latencies), 2),
        "avg_cost": round(sum(costs)     / len(costs),     6),
    }

In [ ]:
# Evaluating results with Retriever-Specific Ragas Metrics!

import json
import time
from openai import OpenAI

oai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def ask_llm(prompt):
    """Single synchronous OpenAI call with a short sleep to avoid rate limits."""
    time.sleep(0.5)
    response = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10,
    )
    return response.choices[0].message.content.strip().lower()

def score_context_precision(question, contexts, reference):
    """What fraction of retrieved chunks are actually relevant?"""
    relevant = 0
    for chunk in contexts:
        prompt = f"""Given this question: "{question}"
Is the following context chunk relevant to answering it?
Chunk: "{chunk[:500]}"
Answer with only yes or no."""
        answer = ask_llm(prompt)
        if "yes" in answer:
            relevant += 1
    return relevant / len(contexts) if contexts else 0

def score_context_recall(question, contexts, reference):
    """Does the retrieved context cover the reference answer?"""
    combined_context = " ".join(contexts)[:2000]
    prompt = f"""Given this reference answer: "{reference[:500]}"
And this retrieved context: "{combined_context}"
Does the retrieved context contain enough information to produce the reference answer?
Answer with only yes or no."""
    answer = ask_llm(prompt)
    return 1.0 if "yes" in answer else 0.0

def score_context_entity_recall(question, contexts, reference):
    """Are the key entities from the reference present in the context?"""
    combined_context = " ".join(contexts)[:2000]
    prompt = f"""Reference answer: "{reference[:500]}"
Retrieved context: "{combined_context}"
Are the main facts, numbers, and named concepts from the reference answer present in the retrieved context?
Answer with only yes or no."""
    answer = ask_llm(prompt)
    return 1.0 if "yes" in answer else 0.0

def score_noise_sensitivity(question, contexts, response):
    """Does the answer stay accurate despite potentially irrelevant chunks?"""
    combined_context = " ".join(contexts)[:2000]
    prompt = f"""Question: "{question}"
Retrieved context: "{combined_context}"
Generated answer: "{response[:500]}"
Does the generated answer contain any claims that are NOT supported by the retrieved context?
Answer with only yes or no."""
    answer = ask_llm(prompt)
    # noise sensitivity: yes = answer has unsupported claims = bad = low score
    return 0.0 if "yes" in answer else 1.0

# ── Run evaluation ────────────────────────────────────────────────────────────
ragas_scores = {}

for name, rec in run_results.items():
    print(f"\nEvaluating: {name}")
    precision_scores, recall_scores, entity_scores, noise_scores = [], [], [], []

    for i in range(len(questions)):
        q   = questions[i]
        ctx = rec["contexts"][i]
        ref = references[i]
        ans = rec["answers"][i]

        precision_scores.append(score_context_precision(q, ctx, ref))
        recall_scores.append(score_context_recall(q, ctx, ref))
        entity_scores.append(score_context_entity_recall(q, ctx, ref))
        noise_scores.append(score_noise_sensitivity(q, ctx, ans))

        # Print progress every 5 questions so you can see it moving
        if (i + 1) % 5 == 0:
            print(f"  {i + 1}/{len(questions)} questions done")

    ragas_scores[name] = {
        "context_precision":      round(sum(precision_scores) / len(precision_scores), 4),
        "context_recall":         round(sum(recall_scores)    / len(recall_scores),    4),
        "context_entity_recall":  round(sum(entity_scores)    / len(entity_scores),    4),
        "noise_sensitivity":      round(sum(noise_scores)     / len(noise_scores),     4),
    }

    # Print immediately so scores are never lost
    print(f"  ✅ Results for {name}:")
    for metric, val in ragas_scores[name].items():
        print(f"    {metric}: {val}")

print("\nAll evaluation complete.")


Evaluating: Naive
  5/18 questions done
  10/18 questions done
  15/18 questions done
  ✅ Results for Naive:
    context_precision: 0.15
    context_recall: 1.0
    context_entity_recall: 1.0
    noise_sensitivity: 0.8889

Evaluating: BM25
  5/18 questions done
  10/18 questions done
  15/18 questions done
  ✅ Results for BM25:
    context_precision: 0.1528
    context_recall: 0.6667
    context_entity_recall: 0.5556
    noise_sensitivity: 0.6667

Evaluating: Multi-Query
  5/18 questions done
  10/18 questions done
  15/18 questions done
  ✅ Results for Multi-Query:
    context_precision: 0.1328
    context_recall: 1.0
    context_entity_recall: 1.0
    noise_sensitivity: 0.9444

Evaluating: Parent-Document
  5/18 questions done
  10/18 questions done
  15/18 questions done
  ✅ Results for Parent-Document:
    context_precision: 0.2546
    context_recall: 0.9444
    context_entity_recall: 0.9444
    noise_sensitivity: 0.6111

Evaluating: Contextual-Compress
  5/18 questions done
  10/

In [ ]:
# Comparing the results of all five retrievers- heck yeah!

rows = []
for name in retriever_map:
    row = {
        "Retriever":           name,
        "Avg Latency (s)":     run_results[name]["avg_lat"],
        "Est. Cost/Query ($)": run_results[name]["avg_cost"],
    }
    row.update({k: round(v, 4) for k, v in ragas_scores[name].items()})
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("Retriever")

display(
    results_df.style
        .highlight_max(axis=0,
            subset=[c for c in results_df.columns if c not in ["Avg Latency (s)", "Est. Cost/Query ($)"]],
            color="#c6efce")
        .highlight_min(axis=0,
            subset=["Avg Latency (s)", "Est. Cost/Query ($)"],
            color="#c6efce")
        .format(precision=4)
        .set_caption("Retriever Comparison — green = best per column")
)

,Avg Latency (s),Est. Cost/Query ($),context_precision,context_recall,context_entity_recall,noise_sensitivity
Retriever,,,,,,
Naive,2.9500,0.0003,0.1500,1.0000,1.0000,0.8889
BM25,2.2800,0.0002,0.1528,0.6667,0.5556,0.6667
Multi-Query,3.7500,0.0004,0.1328,1.0000,1.0000,0.9444
Parent-Document,2.7800,0.0003,0.2546,0.9444,0.9444,0.6111
Contextual-Compress,2.5500,0.0002,0.4074,1.0000,1.0000,1.0000


**SUMMARY RESULTS**

All five retrievers were evaluated using the Health & Wellness Guide- 

Contextual Compression was the standout performer. It achieved the highest context precision (0.41), perfect recall and entity recall (1.0), and perfect noise sensitivity (1.0), while tying for the lowest cost per query ($0.0002) and posting a competitive latency of 2.55s. 

The Cohere reranker had the ability to filter retrieved chunks down to only the most relevant passages, making it highly effective on this dataset because its way topic-dense! 

Naive Dense matched on recall and entity recall (1.0) with good noise sensitivity (0.89)  and at a similarly low cost, which makes it the strongest contender if Cohere is unavailable. 

Multi-Query also achieved excellent recall and entity recall but at the highest latency and cost — the extra LLM call for query expansion retrieves over a broad range which is good, but it does so imprecisely (0.13 precision).

 Parent-Document generated the best precision of the non-compression retrievers (0.25) but had the worst noise sensitivity (0.61), confirming that large parent chunks bring useful context but they also introduce unwanted/irrelevant off-topic content. 
 
 BM25 was the fastest and cheapest retriever, but sadly had the weakest results across every performance metric, struggling with paraphrased conversational questions, which are super common in this dataset. 
 
 At the end of the day, Contextual Compression is unquestionably the best choice of retriever in the case of this set of data, with Naive Dense as a solid default backup if deployment is cost-constrained.




![Retriever Comparison](retriever_comparison.png)